# Drive VGGT Runner

Clean Colab runner for a `cloud_vggt_job.zip` stored in Google Drive. It does not use the upload widget.

Default input is the big package: `cloud_vggt_job.zip`. To run the smaller package later, change `JOB_ZIP_NAME` to `cloud_vggt_job_small.zip` in the config cell.

Safety labels for the returned artifacts: no geolocation, no meters, relative VGGT frame, local-only media-derived frames.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

# Run the big package first. Change this to cloud_vggt_job_small.zip later if needed.
JOB_ZIP_NAME = "cloud_vggt_job.zip"

DRIVE_ROOT = Path("/content/drive/MyDrive")
WORK_ROOT = Path("/content/fpv_vggt_drive_run")
RETURN_ROOT = DRIVE_ROOT / "fpv_vggt_returns"

print("Mounting Google Drive...")
drive.mount("/content/drive")
print("Input ZIP name:", JOB_ZIP_NAME)
print("Drive root:", DRIVE_ROOT)
print("Work root:", WORK_ROOT)
print("Return root:", RETURN_ROOT)

In [ ]:
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, select a T4 GPU runtime.")

props = torch.cuda.get_device_properties(0)
memory_gib = props.total_memory / (1024 ** 3)
print("CUDA device:", props.name)
print(f"GPU memory: {memory_gib:.1f} GiB")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
def find_drive_zip(root: Path, name: str) -> Path:
    direct = root / name
    if direct.exists():
        return direct
    matches = sorted(root.rglob(name), key=lambda path: (len(path.parts), str(path)))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} anywhere under {root}")
    if len(matches) > 1:
        print("Multiple matching ZIPs found. Using the first one:")
        for match in matches:
            print(" -", match)
    return matches[0]

source_zip = find_drive_zip(DRIVE_ROOT, JOB_ZIP_NAME)
print("Using ZIP:", source_zip)
print(f"ZIP size: {source_zip.stat().st_size / (1024 * 1024):.2f} MiB")

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(source_zip) as archive:
    archive.extractall(WORK_ROOT)

candidates = sorted(WORK_ROOT.rglob("run_vggt_job.py"))
if not candidates:
    raise FileNotFoundError("run_vggt_job.py was not found inside the ZIP.")

JOB_DIR = candidates[0].parent
manifest_path = JOB_DIR / "job_manifest.json"
manifest = json.loads(manifest_path.read_text())

print("Job dir:", JOB_DIR)
print("Clip count:", len(manifest.get("clips", [])))
for clip in manifest.get("clips", []):
    print("-", clip["clip_id"], "frames=", clip.get("frame_count"))

print("Warnings:")
for warning in manifest.get("warnings", []):
    print("-", warning)

In [ ]:
def run_job(command, cwd: Path) -> None:
    print("+", " ".join(str(part) for part in command))
    start = time.time()
    try:
        subprocess.run([str(part) for part in command], cwd=str(cwd), check=True)
    except subprocess.CalledProcessError:
        log_path = cwd / "cloud_run.log"
        if log_path.exists():
            print("\nLast cloud_run.log lines:")
            print("\n".join(log_path.read_text(errors="replace").splitlines()[-80:]))
        raise
    finally:
        elapsed_min = (time.time() - start) / 60
        print(f"Elapsed: {elapsed_min:.1f} minutes")

run_job([sys.executable, "run_vggt_job.py"], JOB_DIR)

In [ ]:
import json
from pathlib import Path

print("JOB_DIR:", JOB_DIR)
print("JOB_ZIP_NAME:", JOB_ZIP_NAME)
print("source_zip:", source_zip)

runner_path = JOB_DIR / "run_vggt_job.py"
runner_text = runner_path.read_text(errors="replace")
print("runner:", runner_path)
print("fixed runner present:", "def to_numpy_array(value):" in runner_text)

log_path = JOB_DIR / "cloud_run.log"
summary_path = JOB_DIR / "cloud_summary.json"

print("\ncloud_run.log exists:", log_path.exists())
if log_path.exists():
    print("\n--- LAST 120 LOG LINES ---")
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-120:]))

print("\ncloud_summary.json exists:", summary_path.exists())
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("\n--- SUMMARY STATUS ---")
    print("status:", summary.get("status"))
    print("dependency_error:", summary.get("dependency_error"))

    for clip in summary.get("clips", []):
        print("\n--- CLIP ---")
        print("clip_id:", clip.get("clip_id"))
        print("status:", clip.get("status"))
        print("bundle_valid:", clip.get("bundle_valid"))
        print("error:", clip.get("error"))
        tb = clip.get("traceback")
        if tb:
            print("traceback tail:")
            print("\n".join(tb.splitlines()[-30:]))

In [ ]:
summary_path = JOB_DIR / "cloud_summary.json"
log_path = JOB_DIR / "cloud_run.log"
bundles_path = JOB_DIR / "bundles.zip"

if not summary_path.exists():
    raise FileNotFoundError("cloud_summary.json was not created.")
if not bundles_path.exists():
    raise FileNotFoundError("bundles.zip was not created. The VGGT job did not finish successfully.")

summary = json.loads(summary_path.read_text())
print("Cloud summary status:", summary.get("status"))
for clip in summary.get("clips", []):
    print("-", clip.get("clip_id"), "status=", clip.get("status"), "bundle_valid=", clip.get("bundle_valid"))

return_dir = RETURN_ROOT / source_zip.stem
return_dir.mkdir(parents=True, exist_ok=True)
for path in [bundles_path, summary_path, log_path]:
    if path.exists():
        destination = return_dir / path.name
        shutil.copy2(path, destination)
        print("Copied:", destination)

print("\nReturn directory in Drive:", return_dir)
print("Download or sync bundles.zip back to the local repo, then import it locally with:")
print("fpv vggt import-cloud-job --source <returned-bundles.zip> --output-root data/vggt --report outputs/reviews/cloud_bundle_import.json")

## After Colab Finishes

The returned files are copied to Google Drive under:

```text
My Drive/fpv_vggt_returns/cloud_vggt_job/
```

Bring `bundles.zip` back to the local project and import it with:

```bash
fpv vggt import-cloud-job --source <returned-bundles.zip> --output-root data/vggt --report outputs/reviews/cloud_bundle_import.json
```

If the big package runs out of memory, change `JOB_ZIP_NAME` to `cloud_vggt_job_small.zip`, restart the runtime, and rerun the notebook.